<a href="https://colab.research.google.com/github/aistudent4321-arch/data-mining-recommender/blob/main/Last_Update_Data_Mining_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlxtend scikit-learn pandas numpy scipy


#  Imports

In [2]:
import pandas as pd
import numpy as np
import warnings
import itertools
from collections import defaultdict, Counter
from datetime import datetime

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.stats import chi2_contingency

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

print("✅ All libraries imported successfully.")
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Device name     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ All libraries imported successfully.
PyTorch version : 2.10.0+cu128
GPU available   : True
Device name     : Tesla T4


# Load All Data




In [3]:
from google.colab import files
import pandas as pd
import io

print("ارفع الملفات الستة الآن:")
uploaded = files.upload()  # ← هيطلب منك تختار الملفات

train_transactions = pd.read_csv(io.BytesIO(uploaded["train_transactions.csv"]))
train_baskets      = pd.read_csv(io.BytesIO(uploaded["train_baskets.csv"]))
train_users        = pd.read_csv(io.BytesIO(uploaded["train_users.csv"]))
menu_items         = pd.read_csv(io.BytesIO(uploaded["menu_items.csv"]))
test_instances     = pd.read_csv(io.BytesIO(uploaded["test_instances.csv"]))
sample_submission  = pd.read_csv(io.BytesIO(uploaded["sample_submission.csv"]))

print("\n✅ Data loaded.")

for name, df in [
    ("train_transactions", train_transactions),
    ("train_baskets",      train_baskets),
    ("train_users",        train_users),
    ("menu_items",         menu_items),
    ("test_instances",     test_instances),
    ("sample_submission",  sample_submission),
]:
    print(f"{name:25s} → {df.shape[0]:>7,} rows × {df.shape[1]:>2} cols")

print("\n✅ Data loaded.")

ارفع الملفات الستة الآن:


Saving sample_submission.csv to sample_submission.csv
Saving test_instances.csv to test_instances.csv
Saving menu_items.csv to menu_items.csv
Saving train_baskets.csv to train_baskets.csv
Saving train_users.csv to train_users.csv
Saving train_transactions.csv to train_transactions.csv

✅ Data loaded.
train_transactions        → 170,793 rows × 16 cols
train_baskets             →  85,486 rows × 16 cols
train_users               →  10,000 rows ×  3 cols
menu_items                →      50 rows × 15 cols
test_instances            →  14,147 rows ×  8 cols
sample_submission         →  14,147 rows × 11 cols

✅ Data loaded.


# Preprocessing




In [4]:
for df in [train_transactions, train_baskets]:
    df["timestamp"] = pd.to_datetime(df["timestamp"])

train_transactions.sort_values("timestamp", inplace=True)
train_transactions.reset_index(drop=True, inplace=True)

train_baskets.sort_values("timestamp", inplace=True)
train_baskets.reset_index(drop=True, inplace=True)

train_transactions.drop_duplicates(inplace=True)
train_baskets.drop_duplicates(inplace=True)

def engineer_features(df):
    df = df.copy()
    df["hour"]        = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.day_name()
    df["is_weekend"]  = df["timestamp"].dt.dayofweek.isin([4, 5]).astype(int)
    df["month"]       = df["timestamp"].dt.month
    df["year"]        = df["timestamp"].dt.year

    def assign_meal_period(h):
        if 5 <= h < 12:
            return "breakfast"
        elif 12 <= h < 17:
            return "lunch"
        elif 17 <= h < 21:
            return "dinner"
        else:
            return "late_night"

    df["meal_period"] = df["hour"].apply(assign_meal_period)
    return df

train_transactions = engineer_features(train_transactions)
train_baskets      = engineer_features(train_baskets)

def parse_items(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return []
    return [x.strip() for x in str(raw).split("|") if x.strip() != ""]

train_baskets["item_list"]       = train_baskets["item_ids"].apply(parse_items)
test_instances["basket_list"]    = test_instances["current_basket_items"].apply(parse_items)

ALL_ITEMS = menu_items["item_id"].tolist()
ITEM_SET  = set(ALL_ITEMS)

max_ts    = train_transactions["timestamp"].max()
min_ts    = train_transactions["timestamp"].min()
span_days = max(1, (max_ts - min_ts).days)

train_transactions["days_ago"] = (max_ts - train_transactions["timestamp"]).dt.days

HALF_LIFE = 90
train_transactions["recency_weight"] = np.exp(
    -np.log(2) * train_transactions["days_ago"] / HALF_LIFE
)

train_baskets["days_ago"] = (max_ts - train_baskets["timestamp"]).dt.days
train_baskets["recency_weight"] = np.exp(
    -np.log(2) * train_baskets["days_ago"] / HALF_LIFE
)

print(f"Transactions : {len(train_transactions):,}")
print(f"Baskets      : {len(train_baskets):,}")
print(f"Date range   : {min_ts.date()} → {max_ts.date()}")
print(f"Menu items   : {len(ALL_ITEMS)}")
print("\n✅ Preprocessing complete.")

Transactions : 170,793
Baskets      : 85,486
Date range   : 2024-01-01 → 2025-07-01
Menu items   : 50

✅ Preprocessing complete.


# Train / Validation Split (Time-Based)



In [5]:
VALIDATION_FRACTION = 0.10

cutoff_idx  = int(len(train_baskets) * (1 - VALIDATION_FRACTION))
cutoff_time = train_baskets.iloc[cutoff_idx]["timestamp"]

val_baskets      = train_baskets[train_baskets["timestamp"] >= cutoff_time].copy()
train_baskets_tr = train_baskets[train_baskets["timestamp"] <  cutoff_time].copy()

val_transactions = train_transactions[train_transactions["timestamp"] >= cutoff_time].copy()
train_tx_tr      = train_transactions[train_transactions["timestamp"] <  cutoff_time].copy()

print(f"Cut-off date         : {cutoff_time.date()}")
print(f"Train baskets        : {len(train_baskets_tr):,}")
print(f"Validation baskets   : {len(val_baskets):,}")
print(f"Train transactions   : {len(train_tx_tr):,}")
print(f"Validation tx        : {len(val_transactions):,}")

val_instances_list = []
for _, row in val_baskets.iterrows():
    items = row["item_list"]
    if len(items) < 2:
        continue
    context  = items[:-1]
    held_out = items[-1]
    val_instances_list.append({
        "user_id":     row["user_id"],
        "timestamp":   row["timestamp"],
        "meal_period": row["meal_period"],
        "is_weekend":  row["is_weekend"],
        "hour":        row["hour"],
        "context":     context,
        "held_out":    held_out,
    })

val_df = pd.DataFrame(val_instances_list)
print(f"\nValidation prediction instances : {len(val_df):,}")
print("\n✅ Time-based split complete.")

Cut-off date         : 2025-05-08
Train baskets        : 76,937
Validation baskets   : 8,549
Train transactions   : 153,686
Validation tx        : 17,107

Validation prediction instances : 5,906

✅ Time-based split complete.


# Helper Functions

In [6]:
def ndcg_at_k(recommended: list, relevant: str, k: int = 10) -> float:
    for rank, item in enumerate(recommended[:k], start=1):
        if item == relevant:
            return 1.0 / np.log2(rank + 1)
    return 0.0


def evaluate_ndcg(predictions: dict, val_df: pd.DataFrame, k: int = 10) -> float:
    scores = []
    for _, row in val_df.iterrows():
        key   = (row["user_id"], row["timestamp"])
        preds = predictions.get(key, [])
        scores.append(ndcg_at_k(preds, row["held_out"], k))
    return float(np.mean(scores))


def remove_basket_items(scores: dict, basket: list) -> dict:
    for item in basket:
        if item in scores:
            scores[item] = 0.0
    return scores


def top_k_items(scores: dict, k: int = 10, fallback: list = None) -> list:
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    result = [item for item, score in ranked if score > 0][:k]

    if fallback and len(result) < k:
        for item in fallback:
            if item not in result:
                result.append(item)
            if len(result) == k:
                break

    for item in ALL_ITEMS:
        if len(result) >= k:
            break
        if item not in result:
            result.append(item)

    return result[:k]


def normalize_scores(scores: dict) -> dict:
    if not scores:
        return scores
    min_v = min(scores.values())
    max_v = max(scores.values())
    rng   = max_v - min_v
    if rng == 0:
        return {k: 1.0 for k in scores}
    return {k: (v - min_v) / rng for k, v in scores.items()}


def empty_scores() -> dict:
    return {item: 0.0 for item in ALL_ITEMS}


print("✅ Helper functions defined.")

✅ Helper functions defined.


# Binary / Weighted User-Item Matrix

In [7]:
agg = (
    train_tx_tr
    .groupby(["user_id", "item_id"])
    .agg(
        freq        = ("item_id",        "count"),
        recency_sum = ("recency_weight",  "sum"),
    )
    .reset_index()
)

freq_matrix = agg.pivot(index="user_id", columns="item_id", values="freq").fillna(0)
wt_matrix   = agg.pivot(index="user_id", columns="item_id", values="recency_sum").fillna(0)

for item in ALL_ITEMS:
    if item not in freq_matrix.columns:
        freq_matrix[item] = 0.0
    if item not in wt_matrix.columns:
        wt_matrix[item] = 0.0

freq_matrix   = freq_matrix[ALL_ITEMS]
wt_matrix     = wt_matrix[ALL_ITEMS]
binary_matrix = (freq_matrix > 0).astype(float)

print(f"User-item matrix shape : {freq_matrix.shape}")
print(f"Sparsity               : {100*(1 - (freq_matrix > 0).values.mean()):.1f}%")
print(f"Max frequency cell     : {int(freq_matrix.values.max())}")
print(f"\n✅ Matrices built.")

User-item matrix shape : (8866, 50)
Sparsity               : 85.8%
Max frequency cell     : 60

✅ Matrices built.


# User Vectorization

In [8]:
item_meta = menu_items.set_index("item_id")[
    ["category", "base_price", "is_spicy", "is_vegetarian", "is_vegan"]
].to_dict(orient="index")

tx = train_tx_tr.copy()
tx["base_price"]    = tx["item_id"].map(lambda x: item_meta.get(x, {}).get("base_price", 0))
tx["is_spicy"]      = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_spicy", False)))
tx["is_vegetarian"] = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_vegetarian", False)))
tx["is_vegan"]      = tx["item_id"].map(lambda x: int(item_meta.get(x, {}).get("is_vegan", False)))
tx["category"]      = tx["item_id"].map(lambda x: item_meta.get(x, {}).get("category", "Unknown"))

ALL_CATEGORIES = menu_items["category"].unique().tolist()

def build_user_vectors(tx_df):
    user_records = []
    for user_id, grp in tx_df.groupby("user_id"):
        avg_price        = grp["base_price"].mean()
        spicy_ratio      = grp["is_spicy"].mean()
        vegetarian_ratio = grp["is_vegetarian"].mean()
        vegan_ratio      = grp["is_vegan"].mean()

        period_counts = grp["meal_period"].value_counts(normalize=True)
        breakfast_r   = period_counts.get("breakfast",  0.0)
        lunch_r       = period_counts.get("lunch",      0.0)
        dinner_r      = period_counts.get("dinner",     0.0)
        late_r        = period_counts.get("late_night", 0.0)

        weekend_ratio = grp["is_weekend"].mean()
        cat_counts    = grp["category"].value_counts(normalize=True)
        cat_feats     = {f"cat_{c}": cat_counts.get(c, 0.0) for c in ALL_CATEGORIES}

        record = {
            "user_id":          user_id,
            "avg_price":        avg_price,
            "spicy_ratio":      spicy_ratio,
            "vegetarian_ratio": vegetarian_ratio,
            "vegan_ratio":      vegan_ratio,
            "breakfast_ratio":  breakfast_r,
            "lunch_ratio":      lunch_r,
            "dinner_ratio":     dinner_r,
            "late_night_ratio": late_r,
            "weekend_ratio":    weekend_ratio,
            **cat_feats,
        }
        user_records.append(record)

    df = pd.DataFrame(user_records).set_index("user_id").fillna(0)
    return df

user_vectors = build_user_vectors(tx)

print(f"User vectors shape : {user_vectors.shape}")
print(f"Feature columns    : {list(user_vectors.columns)[:8]} ...")
print("\n✅ User vectors built.")

User vectors shape : (8866, 18)
Feature columns    : ['avg_price', 'spicy_ratio', 'vegetarian_ratio', 'vegan_ratio', 'breakfast_ratio', 'lunch_ratio', 'dinner_ratio', 'late_night_ratio'] ...

✅ User vectors built.


# Item-Item Similarity Model..





In [9]:
item_vectors     = wt_matrix.T.values
item_sim_matrix  = cosine_similarity(item_vectors)
item_ids_ordered = wt_matrix.columns.tolist()
item_idx         = {item: i for i, item in enumerate(item_ids_ordered)}

def item_similarity_scores(seed_items: list, top_n: int = 30) -> dict:
    scores       = empty_scores()
    valid_seeds  = [s for s in seed_items if s in item_idx]
    if not valid_seeds:
        return scores
    for seed in valid_seeds:
        idx     = item_idx[seed]
        sim_row = item_sim_matrix[idx]
        for j, item in enumerate(item_ids_ordered):
            scores[item] = max(scores.get(item, 0.0), float(sim_row[j]))
    for s in valid_seeds:
        scores[s] = 0.0
    return scores

binary_vals     = binary_matrix.values

def jaccard_sim_matrix(bin_mat):
    n_items      = bin_mat.shape[1]
    intersection = bin_mat.T @ bin_mat
    col_sums     = bin_mat.sum(axis=0)
    union        = col_sums[:, None] + col_sums[None, :] - intersection
    with np.errstate(divide="ignore", invalid="ignore"):
        jac = np.where(union == 0, 0.0, intersection / union)
    return jac

item_jac_matrix = jaccard_sim_matrix(binary_vals)

print(f"Cosine sim matrix shape  : {item_sim_matrix.shape}")
print(f"Jaccard sim matrix shape : {item_jac_matrix.shape}")

if "WRAP001" in item_idx:
    idx0 = item_idx["WRAP001"]
    top5 = np.argsort(item_sim_matrix[idx0])[::-1][1:6]
    print("\nTop-5 cosine-similar items to WRAP001:")
    for r in top5:
        print(f"  {item_ids_ordered[r]:12s}  sim={item_sim_matrix[idx0][r]:.4f}")

print("\n✅ Item-item similarity models built.")

Cosine sim matrix shape  : (50, 50)
Jaccard sim matrix shape : (50, 50)

Top-5 cosine-similar items to WRAP001:
  SAUCE001      sim=0.8365
  SIDE001       sim=0.4765
  WRAP004       sim=0.4386
  BEV006        sim=0.4303
  WRAP002       sim=0.4066

✅ Item-item similarity models built.


# Cell 9 - Collaborative Filtering


In [10]:
from sklearn.neighbors import NearestNeighbors

def item_based_cf_scores(user_id: str, basket: list, top_n: int = 30) -> dict:
    scores = empty_scores()
    if user_id not in wt_matrix.index:
        return item_similarity_scores(basket)
    user_row = wt_matrix.loc[user_id].values
    for j, target_item in enumerate(item_ids_ordered):
        if target_item in ITEM_SET:
            val = float(user_row @ item_sim_matrix[:, j])
            scores[target_item] = val
    return scores


USER_VEC_MATRIX = user_vectors.values
USER_VEC_INDEX  = {uid: i for i, uid in enumerate(user_vectors.index)}

# ── Approximate KNN بدل cosine matrix الكاملة ──────────────────────────────
N_NEIGHBORS = 30
print(f"Building Approximate User-KNN (n_neighbors={N_NEIGHBORS}) ...")
knn_model = NearestNeighbors(
    n_neighbors = N_NEIGHBORS + 1,
    metric      = "cosine",
    algorithm   = "brute",
    n_jobs      = -1,
)
knn_model.fit(USER_VEC_MATRIX)
print("✅ KNN model ready")


def user_based_cf_scores(user_id: str, k_neighbors: int = N_NEIGHBORS) -> dict:
    scores = empty_scores()
    if user_id not in USER_VEC_INDEX:
        return scores
    u_idx = USER_VEC_INDEX[user_id]
    u_vec = USER_VEC_MATRIX[u_idx].reshape(1, -1)

    dists, idxs = knn_model.kneighbors(u_vec, n_neighbors=k_neighbors + 1)

    for dist, n_idx in zip(dists[0][1:], idxs[0][1:]):
        sim_val      = 1.0 - dist
        neighbor_uid = user_vectors.index[n_idx]
        if sim_val <= 0 or neighbor_uid not in wt_matrix.index:
            continue
        neighbor_row = wt_matrix.loc[neighbor_uid].values
        for j, item in enumerate(item_ids_ordered):
            scores[item] += sim_val * neighbor_row[j]
    return scores


def user_history_score(user_id: str) -> dict:
    scores = empty_scores()
    if user_id in wt_matrix.index:
        row = wt_matrix.loc[user_id]
        for item in ALL_ITEMS:
            scores[item] = float(row.get(item, 0.0))
    return scores


print("\n✅ Collaborative filtering functions ready.")

Building Approximate User-KNN (n_neighbors=30) ...
✅ KNN model ready

✅ Collaborative filtering functions ready.


 # Cell 10 - Frequent Patterns & Association Rules ..


In [11]:
basket_transactions = train_baskets_tr["item_list"].tolist()
basket_transactions = [b for b in basket_transactions if len(b) >= 2]

print(f"Baskets for FP-Growth : {len(basket_transactions):,}")

te       = TransactionEncoder()
te_array = te.fit_transform(basket_transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

MIN_SUPPORT = 0.002  # ← كان 0.005

print(f"Running FP-Growth with min_support={MIN_SUPPORT} ...")
frequent_itemsets           = fpgrowth(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print(f"Frequent itemsets found : {len(frequent_itemsets):,}")
print(frequent_itemsets["length"].value_counts().sort_index().to_string())

rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

print(f"\nAssociation rules found : {len(rules):,}")
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10).to_string())

rule_lookup = defaultdict(list)
for _, row in rules.iterrows():
    ant   = frozenset(row["antecedents"])
    cons  = list(row["consequents"])
    score = float(row["confidence"] * row["lift"])
    for c in cons:
        rule_lookup[ant].append((c, score))

for k in rule_lookup:
    rule_lookup[k].sort(key=lambda x: x[1], reverse=True)

print(f"\nUnique antecedent sets in lookup : {len(rule_lookup):,}")
print("\n✅ Association rules ready.")

Baskets for FP-Growth : 53,193
Running FP-Growth with min_support=0.002 ...
Frequent itemsets found : 222
length
1     38
2    134
3     48
4      2

Association rules found : 182
          antecedents         consequents   support  confidence       lift
0          (SAUCE002)           (WRAP005)  0.003478    0.506849  32.365949
1           (WRAP005)          (SAUCE002)  0.003478    0.222089  32.365949
2          (SAUCE003)           (HIGH002)  0.005245    0.160529   9.352724
3           (HIGH002)          (SAUCE003)  0.005245    0.305586   9.352724
4          (PLATE004)            (APP006)  0.002350    0.090843   7.806483
5            (APP006)          (PLATE004)  0.002350    0.201939   7.806483
6          (PLATE001)  (SAUCE001, APP001)  0.002388    0.033562   7.438679
7  (SAUCE001, APP001)          (PLATE001)  0.002388    0.529167   7.438679
8          (PLATE002)            (APP006)  0.002181    0.083815   7.202541
9            (APP006)          (PLATE002)  0.002181    0.187399   7.20

# Cell 11 - Closed & Maximal Frequent Itemsets


In [12]:
def get_closed_itemsets(fi_df: pd.DataFrame) -> pd.DataFrame:
    closed = []
    rows   = fi_df.to_dict(orient="records")
    for i, row_i in enumerate(rows):
        itemset_i = row_i["itemsets"]
        support_i = row_i["support"]
        is_closed = True
        for j, row_j in enumerate(rows):
            if i == j:
                continue
            if itemset_i < row_j["itemsets"] and abs(row_j["support"] - support_i) < 1e-9:
                is_closed = False
                break
        if is_closed:
            closed.append(row_i)
    return pd.DataFrame(closed).reset_index(drop=True)


def get_maximal_itemsets(fi_df: pd.DataFrame) -> pd.DataFrame:
    maximal = []
    rows    = fi_df.to_dict(orient="records")
    for i, row_i in enumerate(rows):
        itemset_i  = row_i["itemsets"]
        is_maximal = True
        for j, row_j in enumerate(rows):
            if i == j:
                continue
            if itemset_i < row_j["itemsets"]:
                is_maximal = False
                break
        if is_maximal:
            maximal.append(row_i)
    return pd.DataFrame(maximal).reset_index(drop=True)


closed_itemsets  = get_closed_itemsets(frequent_itemsets)
maximal_itemsets = get_maximal_itemsets(frequent_itemsets)

print(f"All frequent itemsets : {len(frequent_itemsets):,}")
print(f"Closed itemsets       : {len(closed_itemsets):,}")
print(f"Maximal itemsets      : {len(maximal_itemsets):,}")

closed_rules = association_rules(
    closed_itemsets,
    metric="lift",
    min_threshold=1.0
)
closed_rules = closed_rules.sort_values("lift", ascending=False).reset_index(drop=True)

closed_rule_lookup = defaultdict(list)
for _, row in closed_rules.iterrows():
    ant   = frozenset(row["antecedents"])
    cons  = list(row["consequents"])
    score = float(row["confidence"] * row["lift"])
    for c in cons:
        closed_rule_lookup[ant].append((c, score))

for k in closed_rule_lookup:
    closed_rule_lookup[k].sort(key=lambda x: x[1], reverse=True)

print(f"\nClosed-rule antecedent sets : {len(closed_rule_lookup):,}")
print(f"\nTop-5 maximal itemsets:")
print(maximal_itemsets.sort_values("support", ascending=False).head(5)[["itemsets", "support"]].to_string())
print("\n✅ Closed & maximal itemsets ready.")

All frequent itemsets : 222
Closed itemsets       : 222
Maximal itemsets      : 122

Closed-rule antecedent sets : 47

Top-5 maximal itemsets:
                        itemsets   support
22  (WRAP001, SAUCE001, HIGH001)  0.012802
18   (BEV007, WRAP001, SAUCE001)  0.012577
23  (SIDE001, SAUCE001, HIGH001)  0.012314
91          (PLATE007, SAUCE001)  0.009663
75            (PLATE002, BEV006)  0.009080

✅ Closed & maximal itemsets ready.


# Cell 12 - Chi-Square & Correlation Analysis


In [13]:
tx_meta = train_tx_tr.copy()
tx_meta["category"] = tx_meta["item_id"].map(
    lambda x: item_meta.get(x, {}).get("category", "Unknown")
)
tx_meta["hour_bin"] = pd.cut(
    tx_meta["hour"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["night", "morning", "afternoon", "evening", "late"],
    right=False
)

def chi2_association(df, col_a, col_b):
    ct               = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, _  = chi2_contingency(ct)
    return chi2, p, dof, ct

print("=" * 60)
print("Chi-Square Tests: Context Features vs Item Category")
print("=" * 60)

for feat in ["meal_period", "is_weekend", "hour_bin"]:
    chi2_val, p_val, dof, ct = chi2_association(tx_meta, feat, "category")
    sig = "✅ significant" if p_val < 0.05 else "❌ not significant"
    print(f"\n{feat:15s} vs category → χ²={chi2_val:,.1f}, p={p_val:.2e}, dof={dof} — {sig}")

context_popularity = {}
for period, grp in tx_meta.groupby("meal_period"):
    counts = grp["item_id"].value_counts()
    context_popularity[period] = (counts / counts.sum()).to_dict()

weekend_popularity = {}
for is_wknd, grp in tx_meta.groupby("is_weekend"):
    counts = grp["item_id"].value_counts()
    weekend_popularity[is_wknd] = (counts / counts.sum()).to_dict()

global_pop_counts = tx_meta["item_id"].value_counts()
global_popularity = (global_pop_counts / global_pop_counts.sum()).to_dict()

print("\n\nContext Popularity Sample (lunch, top-5):")
lunch_pop = sorted(context_popularity.get("lunch", {}).items(), key=lambda x: x[1], reverse=True)
for item, score in lunch_pop[:5]:
    print(f"  {item:12s} {score:.4f}")

hour_price_corr = tx_meta["hour"].corr(tx_meta["unit_price"])
print(f"\nPearson correlation (hour vs unit_price) = {hour_price_corr:.4f}")
print("\n✅ Chi-square analysis & context popularity complete.")

Chi-Square Tests: Context Features vs Item Category

meal_period     vs category → χ²=3,619.6, p=0.00e+00, dof=24 — ✅ significant

is_weekend      vs category → χ²=14.3, p=7.54e-02, dof=8 — ❌ not significant

hour_bin        vs category → χ²=3,752.4, p=0.00e+00, dof=32 — ✅ significant


Context Popularity Sample (lunch, top-5):
  WRAP001      0.3610
  SAUCE001     0.2394
  SIDE001      0.0469
  BEV006       0.0446
  HIGH001      0.0418

Pearson correlation (hour vs unit_price) = -0.0072

✅ Chi-square analysis & context popularity complete.


# Cell 13 - Scoring System


In [23]:
WEIGHTS = {
    "transformer":  0.40,
    "similarity":   0.22,
    "cf_item":      0.13,
    "cf_user":      0.10,
    "context_pop":  0.08,
    "association":  0.05,
    "recency":      0.02,
}

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-6, "Weights must sum to 1!"


def normalize_softmax(scores: dict, temperature: float = 0.1) -> dict:
    if not scores:
        return scores
    items    = list(scores.keys())
    vals     = np.array([scores[i] for i in items], dtype=np.float64)
    vals     = vals - vals.max()
    exp_vals = np.exp(vals / temperature)
    sum_exp  = exp_vals.sum()
    if sum_exp == 0:
        return {i: 1.0 / len(items) for i in items}
    return {items[k]: float(exp_vals[k] / sum_exp) for k in range(len(items))}


def normalize_minmax(scores: dict) -> dict:
    if not scores:
        return scores
    vals  = list(scores.values())
    min_v = min(vals)
    max_v = max(vals)
    rng   = max_v - min_v
    if rng == 0:
        return {k: 0.0 for k in scores}
    return {k: (v - min_v) / rng for k, v in scores.items()}


def association_score(basket: list, lookup: dict = None) -> dict:
    if lookup is None:
        lookup = closed_rule_lookup
    scores     = empty_scores()
    basket_set = frozenset(basket)
    for r in range(len(basket), 0, -1):
        for subset in itertools.combinations(basket, r):
            ant = frozenset(subset)
            if ant in lookup:
                for item, score in lookup[ant]:
                    if item in scores:
                        scores[item] = max(scores[item], score)
    return scores


def recency_score_for_user(user_id: str, last_n: int = 10) -> dict:
    scores = empty_scores()
    if user_id not in train_tx_tr["user_id"].values:
        return scores
    user_tx = (
        train_tx_tr[train_tx_tr["user_id"] == user_id]
        .sort_values("timestamp", ascending=False)
        .head(last_n)
    )
    for rank, (_, row) in enumerate(user_tx.iterrows(), start=1):
        item = row["item_id"]
        if item in scores:
            scores[item] = max(scores[item], 0.9 ** (rank - 1))
    return scores


def context_pop_score(meal_period: str, is_weekend: int) -> dict:
    scores       = empty_scores()
    period_dict  = context_popularity.get(meal_period, global_popularity)
    weekend_dict = weekend_popularity.get(is_weekend,  global_popularity)
    for item in ALL_ITEMS:
        p = period_dict.get(item, 0.0)
        w = weekend_dict.get(item, 0.0)
        scores[item] = 0.7 * p + 0.3 * w
    return scores


def combine_scores(
    user_id:     str,
    basket:      list,
    meal_period: str,
    is_weekend:  int,
    k:           int = 10,
    weights:     dict = None,
    temperature: float = 0.1,
) -> list:
    if weights is None:
        weights = WEIGHTS

    basket_set = set(basket)

    # ← التعديل هنا — يتعامل مع NameError و TypeError معاً
    try:
        raw_trans = transformer_scores(basket, meal_period, is_weekend)
    except NameError:
        raw_trans = empty_scores()
    except TypeError:
        try:
            raw_trans = transformer_scores(basket)
        except Exception:
            raw_trans = empty_scores()

    raw_sim      = item_similarity_scores(basket)
    raw_cf       = item_based_cf_scores(user_id, basket)
    raw_cf_user  = user_based_cf_scores(user_id)
    raw_ctx      = context_pop_score(meal_period, is_weekend)
    raw_assoc    = association_score(basket)
    raw_rec      = recency_score_for_user(user_id)

    n_trans   = normalize_softmax(raw_trans,   temperature)
    n_sim     = normalize_softmax(raw_sim,     temperature)
    n_cf      = normalize_softmax(raw_cf,      temperature)
    n_cf_user = normalize_softmax(raw_cf_user, temperature)
    n_ctx     = normalize_softmax(raw_ctx,     temperature)
    n_assoc   = normalize_minmax(raw_assoc)
    n_rec     = normalize_minmax(raw_rec)

    final = {}
    for item in ALL_ITEMS:
        if item in basket_set:
            continue
        final[item] = (
            weights["transformer"] * n_trans.get(item,   0.0) +
            weights["similarity"]  * n_sim.get(item,     0.0) +
            weights["cf_item"]     * n_cf.get(item,      0.0) +
            weights["cf_user"]     * n_cf_user.get(item, 0.0) +
            weights["context_pop"] * n_ctx.get(item,     0.0) +
            weights["association"] * n_assoc.get(item,   0.0) +
            weights["recency"]     * n_rec.get(item,     0.0)
        )

    if basket:
        for r in range(len(basket), 0, -1):
            for subset in itertools.combinations(basket, r):
                ant = frozenset(subset)
                if ant in closed_rule_lookup:
                    for cons_item, rule_score in closed_rule_lookup[ant]:
                        if cons_item in final:
                            final[cons_item] += 0.03 * min(rule_score, 1.0)

    ranked = sorted(final.items(), key=lambda x: x[1], reverse=True)
    result = [item for item, _ in ranked[:k]]

    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    for item in fallback:
        if len(result) >= k:
            break
        if item not in result and item not in basket_set:
            result.append(item)

    return result[:k]


score_items = combine_scores

print("✅ Scoring system ready.")
print(f"Weights: { {k: round(v, 2) for k, v in WEIGHTS.items()} }")

✅ Scoring system ready.
Weights: {'transformer': 0.4, 'similarity': 0.22, 'cf_item': 0.13, 'cf_user': 0.1, 'context_pop': 0.08, 'association': 0.05, 'recency': 0.02}


# Cell 14 - Internal Evaluation..


In [24]:
# ── تعريف run_eval هنا عشان Cell 14b تشتغل مستقلة ──────────────────────────
EVAL_SAMPLE = min(2000, len(val_df))
val_sample  = val_df.sample(EVAL_SAMPLE, random_state=42).reset_index(drop=True)

def run_eval(score_fn_name: str, score_fn) -> float:
    preds = {}
    for _, row in val_sample.iterrows():
        key  = (row["user_id"], row["timestamp"])
        recs = score_fn(
            row["user_id"],
            row["context"],
            row["meal_period"],
            row["is_weekend"],
        )
        preds[key] = recs
    score = evaluate_ndcg(preds, val_sample)
    print(f"  {score_fn_name:35s} NDCG@10 = {score:.4f}")
    return score
# ─────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM    = 256
N_HEADS      = 8
FF_DIM       = 512
NUM_LAYERS   = 4
MAX_SEQ_LEN  = 20
DROPOUT      = 0.15
EPOCHS       = 20
BATCH_SIZE   = 512

PAD_TOKEN  = "<PAD>"
MASK_TOKEN = "<MASK>"

# ── Context tokens ─────────────────────────────────────────────────────────────
CONTEXT_TOKENS = [
    "<LUNCH>", "<BREAKFAST>", "<DINNER>", "<LATE_NIGHT>",
    "<WEEKEND>", "<WEEKDAY>",
]

MEAL_TOKEN_MAP = {
    "lunch":      "<LUNCH>",
    "breakfast":  "<BREAKFAST>",
    "dinner":     "<DINNER>",
    "late_night": "<LATE_NIGHT>",
}
WEEKEND_TOKEN_MAP = {1: "<WEEKEND>", 0: "<WEEKDAY>"}

vocab    = [PAD_TOKEN, MASK_TOKEN] + CONTEXT_TOKENS + ALL_ITEMS
item2idx = {item: idx for idx, item in enumerate(vocab)}
idx2item = {idx: item for item, idx in item2idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX    = item2idx[PAD_TOKEN]
MASK_IDX   = item2idx[MASK_TOKEN]

print(f"Vocabulary size (with context tokens) : {VOCAB_SIZE}")


# ── Dataset ────────────────────────────────────────────────────────────────────
class MaskedBasketDataset(Dataset):
    def __init__(self, baskets_df, max_len=MAX_SEQ_LEN):
        self.samples = []
        for _, row in baskets_df.iterrows():
            basket = [it for it in row["item_list"] if it in item2idx]
            if len(basket) < 2:
                continue

            mp  = row.get("meal_period", "lunch")
            iw  = int(row.get("is_weekend", 0))
            ctx = [
                MEAL_TOKEN_MAP.get(mp, "<LUNCH>"),
                WEEKEND_TOKEN_MAP.get(iw, "<WEEKDAY>"),
            ]

            max_items = max_len - len(ctx) - 1
            basket    = basket[:max_items]

            mask_pos_in_basket = np.random.randint(0, len(basket))
            target             = item2idx[basket[mask_pos_in_basket]]

            inp = ctx + basket.copy()
            mask_global_pos = len(ctx) + mask_pos_in_basket
            inp[mask_global_pos] = MASK_TOKEN

            inp_idx  = [item2idx[it] for it in inp]
            inp_idx += [PAD_IDX] * (max_len - len(inp_idx))

            self.samples.append((inp_idx, target, mask_global_pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        inp_idx, target, mask_pos = self.samples[idx]
        return (
            torch.tensor(inp_idx,  dtype=torch.long),
            torch.tensor(target,   dtype=torch.long),
            torch.tensor(mask_pos, dtype=torch.long),
        )


train_dataset = MaskedBasketDataset(train_baskets_tr, max_len=MAX_SEQ_LEN)
val_dataset   = MaskedBasketDataset(val_baskets,      max_len=MAX_SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")


# ── Model ──────────────────────────────────────────────────────────────────────
class MaskedBasketTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim  = EMBED_DIM,
        n_heads    = N_HEADS,
        ff_dim     = FF_DIM,
        num_layers = NUM_LAYERS,
        max_len    = MAX_SEQ_LEN,
        dropout    = DROPOUT,
    ):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.drop      = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = n_heads,
            dim_feedforward = ff_dim,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Linear(embed_dim, vocab_size)

        nn.init.xavier_uniform_(self.head.weight)

    def forward(self, x):
        positions    = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        padding_mask = (x == PAD_IDX)
        emb = self.drop(self.embed(x) + self.pos_embed(positions))
        out = self.transformer(emb, src_key_padding_mask=padding_mask)
        out = self.norm(out)
        return self.head(out)


# ── Training ───────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model     = MaskedBasketTransformer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"\nTraining on {DEVICE} for {EPOCHS} epochs ...")
print(f"Model params : {sum(p.numel() for p in model.parameters()):,}")
print("-" * 60)

best_val_loss       = float("inf")
best_model_state    = None
patience_counter    = 0
EARLY_STOP_PATIENCE = 5

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.0
    for inp, target, mask_pos in train_loader:
        inp, target, mask_pos = inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
        logits      = model(inp)
        batch_idx   = torch.arange(inp.size(0), device=DEVICE)
        mask_logits = logits[batch_idx, mask_pos, :]
        loss        = criterion(mask_logits, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    # ── Validation ─────────────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for inp, target, mask_pos in val_loader:
            inp, target, mask_pos = inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
            logits      = model(inp)
            batch_idx   = torch.arange(inp.size(0), device=DEVICE)
            mask_logits = logits[batch_idx, mask_pos, :]
            loss        = criterion(mask_logits, target)
            total_val_loss += loss.item()

    avg_val = total_val_loss / max(len(val_loader), 1)
    scheduler.step()

    gap    = avg_val - avg_train
    status = "  ⚠️  possible overfit" if gap > 0.3 else ""
    print(
        f"  Epoch {epoch:02d}/{EPOCHS} — "
        f"Train: {avg_train:.4f} | "
        f"Val: {avg_val:.4f} | "
        f"Gap: {gap:+.4f}{status}"
    )

    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n  ⏹️  Early stopping at epoch {epoch}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Restored best model (val loss = {best_val_loss:.4f})")

model.eval()


# ── Inference ──────────────────────────────────────────────────────────────────
def transformer_scores(
    basket:      list,
    meal_period: str = "lunch",
    is_weekend:  int = 0,
) -> dict:
    scores = empty_scores()
    known  = [it for it in basket if it in item2idx]

    ctx = [
        MEAL_TOKEN_MAP.get(meal_period, "<LUNCH>"),
        WEEKEND_TOKEN_MAP.get(is_weekend, "<WEEKDAY>"),
    ]

    max_items = MAX_SEQ_LEN - len(ctx) - 1
    if not known:
        seq = ctx + [MASK_TOKEN]
    else:
        seq = ctx + known[-max_items:] + [MASK_TOKEN]

    mask_pos_in_seq = len(seq) - 1

    inp_idx  = [item2idx[it] for it in seq]
    inp_idx += [PAD_IDX] * (MAX_SEQ_LEN - len(inp_idx))

    inp_tensor = torch.tensor([inp_idx], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(inp_tensor)
        probs  = torch.softmax(
            logits[0, mask_pos_in_seq, :], dim=-1
        ).cpu().numpy()

    for item in ALL_ITEMS:
        if item in item2idx:
            scores[item] = float(probs[item2idx[item]])
    return scores


# ── Quick Eval ─────────────────────────────────────────────────────────────────
print("\nEvaluating Transformer on validation sample ...")

def transformer_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = normalize_softmax(transformer_scores(basket, meal_period, is_weekend))
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)

t_ndcg = run_eval("Masked Basket Transformer", transformer_only)
print(f"\nTransformer standalone NDCG@10 : {t_ndcg:.4f}")
print("\n✅ Masked Basket Transformer cell complete.")

Vocabulary size (with context tokens) : 58
Training samples   : 53,193
Validation samples : 5,906

Training on cuda for 20 epochs ...
Model params : 2,143,802
------------------------------------------------------------
  Epoch 01/20 — Train: 2.2037 | Val: 2.0877 | Gap: -0.1161
  Epoch 02/20 — Train: 2.0714 | Val: 2.0789 | Gap: +0.0075
  Epoch 03/20 — Train: 2.0480 | Val: 2.0562 | Gap: +0.0082
  Epoch 04/20 — Train: 2.0390 | Val: 2.0483 | Gap: +0.0092
  Epoch 05/20 — Train: 2.0313 | Val: 2.0470 | Gap: +0.0157
  Epoch 06/20 — Train: 2.0222 | Val: 2.0497 | Gap: +0.0274
  Epoch 07/20 — Train: 2.0188 | Val: 2.0418 | Gap: +0.0230
  Epoch 08/20 — Train: 2.0105 | Val: 2.0421 | Gap: +0.0316
  Epoch 09/20 — Train: 2.0065 | Val: 2.0402 | Gap: +0.0337
  Epoch 10/20 — Train: 2.0020 | Val: 2.0351 | Gap: +0.0331
  Epoch 11/20 — Train: 1.9950 | Val: 2.0464 | Gap: +0.0514
  Epoch 12/20 — Train: 1.9906 | Val: 2.0383 | Gap: +0.0478
  Epoch 13/20 — Train: 1.9844 | Val: 2.0422 | Gap: +0.0578
  Epoch 14/20

#Cell 14b - Masked Basket Transformer

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

PAD_TOKEN  = "<PAD>"
MASK_TOKEN = "<MASK>"

vocab    = [PAD_TOKEN, MASK_TOKEN] + ALL_ITEMS
item2idx = {item: idx for idx, item in enumerate(vocab)}
idx2item = {idx: item for item, idx in item2idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX    = item2idx[PAD_TOKEN]
MASK_IDX   = item2idx[MASK_TOKEN]
MAX_SEQ_LEN = 15

print(f"Vocabulary size : {VOCAB_SIZE}")


class MaskedBasketDataset(Dataset):
    def __init__(self, baskets: list, max_len: int = MAX_SEQ_LEN):
        self.samples = []
        for basket in baskets:
            basket = [it for it in basket if it in item2idx]
            if len(basket) < 2:
                continue
            basket   = basket[:max_len]
            mask_pos = np.random.randint(0, len(basket))
            target   = item2idx[basket[mask_pos]]
            inp      = basket.copy()
            inp[mask_pos] = MASK_TOKEN
            inp_idx  = [item2idx[it] for it in inp]
            inp_idx += [PAD_IDX] * (max_len - len(inp_idx))
            self.samples.append((inp_idx, target, mask_pos))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        inp_idx, target, mask_pos = self.samples[idx]
        return (
            torch.tensor(inp_idx,  dtype=torch.long),
            torch.tensor(target,   dtype=torch.long),
            torch.tensor(mask_pos, dtype=torch.long),
        )


train_basket_lists = train_baskets_tr["item_list"].tolist()
val_basket_lists   = val_baskets["item_list"].tolist()

train_dataset = MaskedBasketDataset(train_basket_lists, max_len=MAX_SEQ_LEN)
val_dataset   = MaskedBasketDataset(val_basket_lists,   max_len=MAX_SEQ_LEN)

train_loader  = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=256, shuffle=False)

print(f"Training samples   : {len(train_dataset):,}")
print(f"Validation samples : {len(val_dataset):,}")


class MaskedBasketTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim  = 128,
        n_heads    = 4,
        ff_dim     = 256,
        num_layers = 2,
        max_len    = MAX_SEQ_LEN,
        dropout    = 0.1,
    ):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model         = embed_dim,
            nhead           = n_heads,
            dim_feedforward = ff_dim,
            dropout         = dropout,
            batch_first     = True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)
        self.head        = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions    = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        padding_mask = (x == PAD_IDX)
        emb = self.embed(x) + self.pos_embed(positions)
        out = self.transformer(emb, src_key_padding_mask=padding_mask)
        out = self.norm(out)
        return self.head(out)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15

model     = MaskedBasketTransformer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

print(f"\nTraining on {DEVICE} for {EPOCHS} epochs ...")
print(f"Model params : {sum(p.numel() for p in model.parameters()):,}")
print("-" * 55)

best_val_loss    = float("inf")
best_model_state = None
patience_counter = 0
EARLY_STOP_PATIENCE = 4

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    for inp, target, mask_pos in train_loader:
        inp, target, mask_pos = (
            inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
        )
        logits      = model(inp)
        batch_idx   = torch.arange(inp.size(0), device=DEVICE)
        mask_logits = logits[batch_idx, mask_pos, :]
        loss        = criterion(mask_logits, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for inp, target, mask_pos in val_loader:
            inp, target, mask_pos = (
                inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
            )
            logits      = model(inp)
            batch_idx   = torch.arange(inp.size(0), device=DEVICE)
            mask_logits = logits[batch_idx, mask_pos, :]
            loss        = criterion(mask_logits, target)
            total_val_loss += loss.item()

    avg_val = total_val_loss / max(len(val_loader), 1)
    scheduler.step(avg_val)

    gap    = avg_val - avg_train
    status = "  ⚠️  possible overfit" if gap > 0.3 else ""
    print(
        f"  Epoch {epoch:02d}/{EPOCHS} — "
        f"Train Loss: {avg_train:.4f} | "
        f"Val Loss: {avg_val:.4f} | "
        f"Gap: {gap:+.4f}{status}"
    )

    if avg_val < best_val_loss:
        best_val_loss    = avg_val
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n  ⏹️  Early stopping at epoch {epoch}")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✅ Restored best model (val loss = {best_val_loss:.4f})")

model.eval()
print("\n✅ Transformer training complete.")


def transformer_scores(basket: list) -> dict:
    scores = empty_scores()
    known  = [it for it in basket if it in item2idx]
    if not known:
        seq = [MASK_TOKEN]
    else:
        seq = known + [MASK_TOKEN]
    seq             = seq[-MAX_SEQ_LEN:]
    mask_pos_in_seq = len(seq) - 1
    inp_idx  = [item2idx[it] for it in seq]
    inp_idx += [PAD_IDX] * (MAX_SEQ_LEN - len(inp_idx))
    inp_tensor = torch.tensor([inp_idx], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        logits = model(inp_tensor)
        probs  = torch.softmax(
            logits[0, mask_pos_in_seq, :], dim=-1
        ).cpu().numpy()
    for item in ALL_ITEMS:
        if item in item2idx:
            scores[item] = float(probs[item2idx[item]])
    return scores


print("\nEvaluating Transformer signal on validation sample ...")

def transformer_only(user_id, basket, meal_period, is_weekend, k=10):
    s        = normalize_softmax(transformer_scores(basket))
    s        = remove_basket_items(s, basket)
    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    return top_k_items(s, k=k, fallback=fallback)

t_ndcg = run_eval("Masked Basket Transformer", transformer_only)
print(f"\nTransformer standalone NDCG@10 : {t_ndcg:.4f}")
print("\n✅ Masked Basket Transformer cell complete.")

Vocabulary size : 52
Training samples   : 53,193
Validation samples : 5,906

Training on cuda for 15 epochs ...
Model params : 280,500
-------------------------------------------------------
  Epoch 01/15 — Train Loss: 2.1138 | Val Loss: 2.0507 | Gap: -0.0632
  Epoch 02/15 — Train Loss: 2.0304 | Val Loss: 2.0393 | Gap: +0.0088
  Epoch 03/15 — Train Loss: 2.0199 | Val Loss: 2.0402 | Gap: +0.0202
  Epoch 04/15 — Train Loss: 2.0126 | Val Loss: 2.0373 | Gap: +0.0246
  Epoch 05/15 — Train Loss: 2.0074 | Val Loss: 2.0298 | Gap: +0.0223
  Epoch 06/15 — Train Loss: 2.0048 | Val Loss: 2.0285 | Gap: +0.0237
  Epoch 07/15 — Train Loss: 2.0034 | Val Loss: 2.0278 | Gap: +0.0245
  Epoch 08/15 — Train Loss: 1.9978 | Val Loss: 2.0265 | Gap: +0.0287
  Epoch 09/15 — Train Loss: 1.9968 | Val Loss: 2.0231 | Gap: +0.0263
  Epoch 10/15 — Train Loss: 1.9939 | Val Loss: 2.0262 | Gap: +0.0323
  Epoch 11/15 — Train Loss: 1.9944 | Val Loss: 2.0256 | Gap: +0.0312
  Epoch 12/15 — Train Loss: 1.9922 | Val Loss: 2.0

In [ ]:
# !! شغّل هذي بعد Cell 14b مباشرة !!
# pip install optuna
!pip install optuna -q

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    w_trans  = trial.suggest_float("transformer",  0.20, 0.60)
    w_sim    = trial.suggest_float("similarity",   0.05, 0.35)
    w_cf     = trial.suggest_float("cf_item",      0.05, 0.25)
    w_cfu    = trial.suggest_float("cf_user",      0.02, 0.20)
    w_ctx    = trial.suggest_float("context_pop",  0.02, 0.20)
    w_assoc  = trial.suggest_float("association",  0.01, 0.15)
    w_rec    = trial.suggest_float("recency",      0.01, 0.10)

    total = w_trans + w_sim + w_cf + w_cfu + w_ctx + w_assoc + w_rec
    trial_weights = {
        "transformer":  w_trans  / total,
        "similarity":   w_sim    / total,
        "cf_item":      w_cf     / total,
        "cf_user":      w_cfu    / total,
        "context_pop":  w_ctx    / total,
        "association":  w_assoc  / total,
        "recency":      w_rec    / total,
    }

    preds = {}
    for _, row in val_sample.iterrows():
        key  = (row["user_id"], row["timestamp"])
        recs = combine_scores(
            user_id     = row["user_id"],
            basket      = row["context"],
            meal_period = row["meal_period"],
            is_weekend  = row["is_weekend"],
            k           = 10,
            weights     = trial_weights,
        )
        preds[key] = recs

    return evaluate_ndcg(preds, val_sample)


print("🔍 Running Optuna weight optimization (100 trials) ...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
total       = sum(best_params.values())

WEIGHTS = {k: v / total for k, v in best_params.items()}

print(f"\n🏆 Best NDCG@10 : {study.best_value:.4f}")
print(f"\n✅ Optimized Weights:")
for k, v in WEIGHTS.items():
    print(f"   {k:15s} : {v:.4f}")

In [1]:
# !! شغّل هذي بعد Optuna !!
!pip install lightgbm -q

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

print("Step 1/4 — Building training features for re-ranker ...")

def build_reranker_features(df, top_k_candidates=30):
    """
    لكل instance يولّد candidates ويحسب features لكل candidate
    """
    rows = []

    for _, row in df.iterrows():
        user_id     = row["user_id"]
        basket      = row["context"]
        meal_period = row["meal_period"]
        is_weekend  = int(row["is_weekend"])
        held_out    = row.get("held_out", None)

        # ── احسب كل الـ scores ──────────────────────────────────────────────
        s_trans = transformer_scores(basket, meal_period, is_weekend)
        s_sim   = normalize_softmax(item_similarity_scores(basket))
        s_cf    = normalize_softmax(item_based_cf_scores(user_id, basket))
        s_cfu   = normalize_softmax(user_based_cf_scores(user_id))
        s_ctx   = normalize_softmax(context_pop_score(meal_period, is_weekend))
        s_assoc = normalize_minmax(association_score(basket))
        s_rec   = normalize_minmax(recency_score_for_user(user_id))
        s_glob  = global_popularity

        # ── ensemble score للحصول على candidates ────────────────────────────
        basket_set = set(basket)
        ensemble   = {}
        for item in ALL_ITEMS:
            if item in basket_set:
                continue
            ensemble[item] = (
                WEIGHTS["transformer"] * s_trans.get(item, 0.0) +
                WEIGHTS["similarity"]  * s_sim.get(item,   0.0) +
                WEIGHTS["cf_item"]     * s_cf.get(item,    0.0) +
                WEIGHTS["cf_user"]     * s_cfu.get(item,   0.0) +
                WEIGHTS["context_pop"] * s_ctx.get(item,   0.0) +
                WEIGHTS["association"] * s_assoc.get(item, 0.0) +
                WEIGHTS["recency"]     * s_rec.get(item,   0.0)
            )

        # أعلى K candidates
        candidates = sorted(ensemble, key=ensemble.get, reverse=True)[:top_k_candidates]

        # تأكد الـ held_out موجود في الـ candidates (للتدريب فقط)
        if held_out and held_out not in candidates:
            candidates = candidates[:-1] + [held_out]

        # ── بناء features لكل candidate ─────────────────────────────────────
        for rank, item in enumerate(candidates):
            meta       = item_meta.get(item, {})
            label      = 1 if item == held_out else 0

            rows.append({
                "user_id":        user_id,
                "item_id":        item,
                "label":          label,

                # Signal scores
                "s_trans":        s_trans.get(item, 0.0),
                "s_sim":          s_sim.get(item,   0.0),
                "s_cf":           s_cf.get(item,    0.0),
                "s_cfu":          s_cfu.get(item,   0.0),
                "s_ctx":          s_ctx.get(item,   0.0),
                "s_assoc":        s_assoc.get(item, 0.0),
                "s_rec":          s_rec.get(item,   0.0),
                "s_glob":         s_glob.get(item,  0.0),
                "s_ensemble":     ensemble.get(item, 0.0),

                # Candidate rank
                "candidate_rank": rank,

                # Item metadata
                "base_price":     meta.get("base_price",    0.0),
                "is_spicy":       int(meta.get("is_spicy",       False)),
                "is_vegetarian":  int(meta.get("is_vegetarian",  False)),
                "is_vegan":       int(meta.get("is_vegan",       False)),

                # Context
                "is_weekend":     is_weekend,
                "meal_period":    meal_period,
                "basket_size":    len(basket),
            })

    return pd.DataFrame(rows)


print("  Building train features ...")
train_feat_df = build_reranker_features(val_sample, top_k_candidates=30)

# Encode meal_period
le = LabelEncoder()
train_feat_df["meal_period_enc"] = le.fit_transform(train_feat_df["meal_period"])

FEATURE_COLS = [
    "s_trans", "s_sim", "s_cf", "s_cfu", "s_ctx",
    "s_assoc", "s_rec", "s_glob", "s_ensemble",
    "candidate_rank", "base_price",
    "is_spicy", "is_vegetarian", "is_vegan",
    "is_weekend", "meal_period_enc", "basket_size",
]

print(f"  Train features shape : {train_feat_df.shape}")
print(f"  Positive labels      : {train_feat_df['label'].sum():,}")

# ── Step 2: Train LightGBM ───────────────────────────────────────────────────
print("\nStep 2/4 — Training LightGBM re-ranker ...")

# Group sizes للـ LambdaRank
group_sizes = train_feat_df.groupby(
    ["user_id", train_feat_df.index // 30]
).size().values

X_train = train_feat_df[FEATURE_COLS].values
y_train = train_feat_df["label"].values

lgb_train = lgb.Dataset(
    X_train,
    label   = y_train,
    group   = group_sizes,
)

lgb_params = {
    "objective":        "lambdarank",
    "metric":           "ndcg",
    "ndcg_eval_at":     [10],
    "learning_rate":    0.05,
    "num_leaves":       63,
    "min_data_in_leaf": 10,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "verbose":          -1,
    "n_jobs":           -1,
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round = 300,
    valid_sets      = [lgb_train],
    callbacks       = [lgb.early_stopping(30), lgb.log_evaluation(50)],
)

print("\n✅ LightGBM re-ranker trained!")

# ── Step 3: Feature Importance ───────────────────────────────────────────────
print("\nStep 3/4 — Feature importance:")
importance = lgb_model.feature_importance(importance_type="gain")
for feat, imp in sorted(zip(FEATURE_COLS, importance), key=lambda x: -x[1]):
    bar = "█" * int(imp / max(importance) * 20)
    print(f"  {feat:20s} {bar} {imp:.1f}")

# ── Step 4: Inference function ───────────────────────────────────────────────
print("\nStep 4/4 — Building re-ranker inference ...")

def reranker_predict(user_id, basket, meal_period, is_weekend, k=10, top_k_candidates=50):
    basket_set = set(basket)

    s_trans = transformer_scores(basket, meal_period, is_weekend)
    s_sim   = normalize_softmax(item_similarity_scores(basket))
    s_cf    = normalize_softmax(item_based_cf_scores(user_id, basket))
    s_cfu   = normalize_softmax(user_based_cf_scores(user_id))
    s_ctx   = normalize_softmax(context_pop_score(meal_period, is_weekend))
    s_assoc = normalize_minmax(association_score(basket))
    s_rec   = normalize_minmax(recency_score_for_user(user_id))
    s_glob  = global_popularity

    ensemble = {}
    for item in ALL_ITEMS:
        if item in basket_set:
            continue
        ensemble[item] = (
            WEIGHTS["transformer"] * s_trans.get(item, 0.0) +
            WEIGHTS["similarity"]  * s_sim.get(item,   0.0) +
            WEIGHTS["cf_item"]     * s_cf.get(item,    0.0) +
            WEIGHTS["cf_user"]     * s_cfu.get(item,   0.0) +
            WEIGHTS["context_pop"] * s_ctx.get(item,   0.0) +
            WEIGHTS["association"] * s_assoc.get(item, 0.0) +
            WEIGHTS["recency"]     * s_rec.get(item,   0.0)
        )

    candidates = sorted(ensemble, key=ensemble.get, reverse=True)[:top_k_candidates]

    meal_enc = le.transform([meal_period])[0] if meal_period in le.classes_ else 0

    feat_rows = []
    for rank, item in enumerate(candidates):
        meta = item_meta.get(item, {})
        feat_rows.append([
            s_trans.get(item, 0.0),
            s_sim.get(item,   0.0),
            s_cf.get(item,    0.0),
            s_cfu.get(item,   0.0),
            s_ctx.get(item,   0.0),
            s_assoc.get(item, 0.0),
            s_rec.get(item,   0.0),
            s_glob.get(item,  0.0),
            ensemble.get(item, 0.0),
            rank,
            meta.get("base_price", 0.0),
            int(meta.get("is_spicy",      False)),
            int(meta.get("is_vegetarian", False)),
            int(meta.get("is_vegan",      False)),
            is_weekend,
            meal_enc,
            len(basket),
        ])

    X_pred  = np.array(feat_rows, dtype=np.float32)
    scores  = lgb_model.predict(X_pred)

    ranked  = sorted(zip(candidates, scores), key=lambda x: -x[1])
    result  = [item for item, _ in ranked[:k]]

    fallback = sorted(global_popularity, key=global_popularity.get, reverse=True)
    for item in fallback:
        if len(result) >= k:
            break
        if item not in result and item not in basket_set:
            result.append(item)

    return result[:k]


# ── Evaluate Re-ranker ───────────────────────────────────────────────────────
print("\nEvaluating Re-ranker on validation sample ...")
reranker_ndcg = run_eval("LightGBM Re-ranker", reranker_predict)
ensemble_ndcg = run_eval("Ensemble only",       lambda u,b,mp,iw,k=10: combine_scores(u,b,mp,iw,k))

print(f"\n{'='*45}")
print(f"  Ensemble NDCG@10    : {ensemble_ndcg:.4f}")
print(f"  Re-ranker NDCG@10   : {reranker_ndcg:.4f}")
print(f"  Improvement         : {reranker_ndcg - ensemble_ndcg:+.4f}")
print(f"{'='*45}")
print("\n✅ LightGBM Re-ranker ready!")

Step 1/4 — Building training features for re-ranker ...
  Building train features ...


NameError: name 'val_sample' is not defined

# Cell 15 Final Prediction Pipeline




In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 15 – Final Prediction Pipeline (Fixed)
# Fixed: transformer_scores uses V2 vocab (no context tokens)
#        because Cell 14b was trained without context tokens
# ══════════════════════════════════════════════════════════════════════════════

print("Step 1/5 – Rebuilding matrices on full training data ...")

# ── Full weighted matrix ───────────────────────────────────────────────────────
agg_full = (
    train_transactions
    .groupby(["user_id", "item_id"])
    .agg(
        freq        = ("item_id",        "count"),
        recency_sum = ("recency_weight",  "sum"),
    )
    .reset_index()
)

wt_matrix_full   = agg_full.pivot(index="user_id", columns="item_id", values="recency_sum").fillna(0)
freq_matrix_full = agg_full.pivot(index="user_id", columns="item_id", values="freq").fillna(0)

for item in ALL_ITEMS:
    if item not in wt_matrix_full.columns:
        wt_matrix_full[item] = 0.0
    if item not in freq_matrix_full.columns:
        freq_matrix_full[item] = 0.0

wt_matrix_full   = wt_matrix_full[ALL_ITEMS]
freq_matrix_full = freq_matrix_full[ALL_ITEMS]
binary_full      = (freq_matrix_full > 0).astype(float)

print(f"  Full wt_matrix shape : {wt_matrix_full.shape}")

# ── Step 2: Rebuild item-item similarity ──────────────────────────────────────
print("Step 2/5 – Rebuilding item-item similarity on full data ...")

item_vectors_full = wt_matrix_full.T.values
item_sim_full     = cosine_similarity(item_vectors_full)
item_ids_full     = wt_matrix_full.columns.tolist()
item_idx_full     = {item: i for i, item in enumerate(item_ids_full)}

print(f"  Item-sim matrix shape : {item_sim_full.shape}")

# ── Step 3: Rebuild context popularity ────────────────────────────────────────
print("Step 3/5 – Rebuilding context popularity on full data ...")

tx_full = train_transactions.copy()
tx_full["category"] = tx_full["item_id"].map(
    lambda x: item_meta.get(x, {}).get("category", "Unknown")
)

ctx_pop_full = {}
for period, grp in tx_full.groupby("meal_period"):
    counts = grp["item_id"].value_counts()
    ctx_pop_full[period] = (counts / counts.sum()).to_dict()

wknd_pop_full = {}
for is_wknd, grp in tx_full.groupby("is_weekend"):
    counts = grp["item_id"].value_counts()
    wknd_pop_full[is_wknd] = (counts / counts.sum()).to_dict()

global_pop_full = tx_full["item_id"].value_counts()
global_pop_full = (global_pop_full / global_pop_full.sum()).to_dict()

print(f"  Context periods found : {list(ctx_pop_full.keys())}")

# ── Step 4: Retrain Transformer on FULL data ──────────────────────────────────
print("Step 4/5 – Retraining Transformer on full basket data ...")

# Use ALL baskets (train + val) — no holdout needed for final model
all_basket_lists = train_baskets["item_list"].tolist()

# Use same MaskedBasketDataset as Cell 14b (V2 — no context tokens)
full_trans_dataset = MaskedBasketDataset(all_basket_lists, max_len=MAX_SEQ_LEN)
full_trans_loader  = DataLoader(full_trans_dataset, batch_size=256, shuffle=True)

print(f"  Full transformer training samples : {len(full_trans_dataset):,}")

# Reinitialize model with same architecture as Cell 14b
model_final     = MaskedBasketTransformer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer_final = optim.Adam(model_final.parameters(), lr=1e-3)
criterion_final = nn.CrossEntropyLoss(label_smoothing=0.1)

# Use same epoch count from Cell 14b
FINAL_EPOCHS = EPOCHS

model_final.train()
for epoch in range(1, FINAL_EPOCHS + 1):
    total_loss = 0.0
    for inp, target, mask_pos in full_trans_loader:
        inp, target, mask_pos = (
            inp.to(DEVICE), target.to(DEVICE), mask_pos.to(DEVICE)
        )
        logits      = model_final(inp)
        batch_idx   = torch.arange(inp.size(0), device=DEVICE)
        mask_logits = logits[batch_idx, mask_pos, :]
        loss        = criterion_final(mask_logits, target)
        optimizer_final.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=1.0)
        optimizer_final.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(full_trans_loader)
    print(f"  Epoch {epoch:02d}/{FINAL_EPOCHS} — Loss: {avg_loss:.4f}")

model_final.eval()
print("  ✅ Final transformer trained on full data.")

# ── Override all in-memory objects with full-data versions ────────────────────
wt_matrix          = wt_matrix_full
item_sim_matrix    = item_sim_full
item_ids_ordered   = item_ids_full
item_idx           = item_idx_full
context_popularity = ctx_pop_full
weekend_popularity = wknd_pop_full
global_popularity  = global_pop_full

# ── Override transformer_scores — V2 style (no context tokens) ───────────────
# IMPORTANT: Cell 14b was trained WITHOUT context tokens (<LUNCH> etc.)
# So we must NOT include them here — KeyError will occur if we do.
# meal_period and is_weekend kept as params for API compatibility only.
def transformer_scores(
    basket:      list,
    meal_period: str = "lunch",
    is_weekend:  int = 0,
) -> dict:
    """
    Inference using the FULL-DATA trained transformer.
    Uses V2 vocab (no context tokens) — matches Cell 14b training.

    Sequence: [item1, item2, ..., <MASK>]
    Empty basket → [<MASK>] alone → model gives global distribution.
    """
    scores = empty_scores()

    known = [it for it in basket if it in item2idx]

    # Empty basket — predict from MASK alone
    if not known:
        seq = [MASK_TOKEN]
    else:
        seq = known + [MASK_TOKEN]

    seq             = seq[-MAX_SEQ_LEN:]
    mask_pos_in_seq = len(seq) - 1

    inp_idx  = [item2idx[it] for it in seq]
    inp_idx += [PAD_IDX] * (MAX_SEQ_LEN - len(inp_idx))

    inp_tensor = torch.tensor([inp_idx], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits = model_final(inp_tensor)
        probs  = torch.softmax(
            logits[0, mask_pos_in_seq, :], dim=-1
        ).cpu().numpy()

    for item in ALL_ITEMS:
        if item in item2idx:
            scores[item] = float(probs[item2idx[item]])

    return scores

print("\nStep 5/5 – Generating predictions for test instances ...")

submission_rows = []

for i, row in test_instances.iterrows():
    instance_id = row["instance_id"]
    user_id     = str(row["user_id"])
    basket      = row["basket_list"]
    meal_period = row["meal_period"]
    is_weekend  = int(row["is_weekend"])

    # ── Use combine_scores as primary function ────────────────────────────────
    # two_stage_recommend also works here if Cell 13 V3 was run
  recs = reranker_predict(
    user_id     = user_id,
    basket      = basket,
    meal_period = meal_period,
    is_weekend  = is_weekend,
    k           = 10,
)

    submission_rows.append({
        "instance_id": instance_id,
        **{f"rank_{j+1}": recs[j] for j in range(10)},
    })

    if (i + 1) % 2000 == 0:
        print(f"  Processed {i+1:,} / {len(test_instances):,} instances ...")

submission_df = pd.DataFrame(submission_rows)

print(f"\nSubmission shape : {submission_df.shape}")
print("\nSample rows:")
print(submission_df.head(3).to_string())
print("\n✅ Predictions generated.")

# Cell 16 - Submission File Generation ..



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 16 – Submission File Generation
# ══════════════════════════════════════════════════════════════════════════════

# ── 16.1  Validate format against sample_submission.csv ──────────────────────
expected_cols = list(sample_submission.columns)
actual_cols   = list(submission_df.columns)

assert actual_cols == expected_cols, (
    f"Column mismatch!\nExpected: {expected_cols}\nGot: {actual_cols}"
)

# ── 16.2  Check all instance_ids are covered ─────────────────────────────────
expected_ids = set(sample_submission["instance_id"])
actual_ids   = set(submission_df["instance_id"])

missing = expected_ids - actual_ids
extra   = actual_ids   - expected_ids

print(f"Expected instances  : {len(expected_ids):,}")
print(f"Submitted instances : {len(actual_ids):,}")
print(f"Missing             : {len(missing)}")
print(f"Extra               : {len(extra)}")

assert len(missing) == 0, f"Missing {len(missing)} instances!"

# ── 16.3  Validate item IDs are all from menu ─────────────────────────────────
rank_cols    = [f"rank_{i}" for i in range(1, 11)]
all_pred_ids = submission_df[rank_cols].values.flatten()
invalid_ids  = set(all_pred_ids) - ITEM_SET

print(f"Invalid item IDs in predictions : {len(invalid_ids)}")
assert len(invalid_ids) == 0, f"Found invalid items: {invalid_ids}"

# ── 16.4  Ensure no duplicates within a row ───────────────────────────────────
def check_row_uniqueness(row):
    items = row[rank_cols].tolist()
    return len(items) == len(set(items))

all_unique = submission_df.apply(check_row_uniqueness, axis=1).all()
print(f"All rows have unique items      : {all_unique}")

# ── 16.5  Reorder rows to match sample_submission order ──────────────────────
submission_df = submission_df.set_index("instance_id")
submission_df = submission_df.loc[sample_submission["instance_id"]]
submission_df = submission_df.reset_index()

# ── 16.6  Save ───────────────────────────────────────────────────────────────
OUTPUT_FILE = "submission.csv"
submission_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n💾 Submission saved to: {OUTPUT_FILE}")
print(f"   Rows : {len(submission_df):,}")
print(f"   Cols : {list(submission_df.columns)}")

print("\nFinal preview:")
print(submission_df.head(5).to_string())

print("\n🎉 Done! Upload submission.csv to the leaderboard.")

In [ ]:
from google.colab import files
files.download("submission.csv")